# Jev-as-a-Judge, LLM-as-a-Judge, and confidence-based escalation

A routing agent predicts an intent. A **judge evaluates that prediction**. In this lab the
judge answers the existing notebook's focused question: **did the router misclassify the
request specifically because the user's utterance was vague?** `TRUE` means that failure
occurred; `FALSE` includes both correct routing and errors caused by something else.

We compare three policies on the same human-labeled traces:

1. Jev alone as the judge.
2. GPT-4.1-mini alone as an independent judge.
3. Jev first, with uncertain or invalid judgments sent to GPT-4.1-mini.

**Replay** recomputes results from a recorded run without a key or new charges. **Live**
mode makes fresh OpenRouter calls and caches them. The notebook selects replay when a
recorded manifest is present, otherwise live. Run every cell in order.

**Benchmark status:** live validation is pending a funded OpenRouter account. No new
Jev/LLM performance numbers or synthetic benchmark results are published in this branch.
This notebook evaluates logs; it does not make calendar changes.

Inspired by the [clinic Jev/GPT notebook](https://colab.research.google.com/drive/1FtSQhtAx8rlkWlV7GA87kIcaHwHG_mL_).
That notebook classifies intents; this one measures agreement with human **judge labels**.

## 1. Why bring System 1 thinking to evaluation?

The System 1 / System 2 analogy separates a quick, narrow judgment from more deliberative
reasoning. Jev applies the first idea to software: it returns a typed decision and
probabilities for a question defined by your policy. An LLM can additionally produce a short
explanation, useful when the case needs closer inspection. These are design roles, not a
claim that either model reproduces human cognition or that the LLM always wins.

A repeated, well-defined check is a useful candidate for a cheap first pass. The hypothesis
we will test is that Jev can handle many cases, while an LLM earns its extra cost on the cases
Jev finds uncertain. Whether that works depends on **error detection, complementary
judgments, and measured costs**. Confidence can be high on an incorrect answer.

See [TypeSafe's System One introduction](https://docs.typesafe.ai/introduction).

In [ ]:
# Run from this directory, the repository root, or a fresh Colab runtime.
import os, sys, subprocess
from pathlib import Path
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_REF = os.environ.get("ROUTING_EVALS_REF", "codex/jev-routing-judge")
candidates = [Path.cwd(), Path.cwd() / "routing_agent_evals",
              Path.cwd() / "AI-Evals-Principles/routing_agent_evals"]
LAB = next((p for p in candidates if (p / "judge_lab.py").exists()), None)
if LAB is None and IN_COLAB:
    checkout = Path("/content/routing-evals-lab")
    if not checkout.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF,
                        "https://github.com/The-Gen-Academy/Mastering-Agentic-AI-Week4-Session1.git",
                        str(checkout)], check=True)
    LAB = checkout / "AI-Evals-Principles/routing_agent_evals"
if LAB is None or not (LAB / "judge_lab.py").exists():
    raise RuntimeError("Open the notebook from its repository checkout; see README for setup.")
LAB = LAB.resolve()
os.chdir(LAB)
sys.path.insert(0, str(LAB))
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "pandas", "scikit-learn", "matplotlib", "python-dotenv"], check=True)

In [ ]:
# Choose replay for the published run, or live for your own experiment.
EVIDENCE = Path(os.environ.get("JUDGE_EVIDENCE_DIR", str(LAB / "data/jev_judge_run")))
DEFAULT_MODE = "replay" if (EVIDENCE / "manifest.json").exists() else "live"
MODE = os.environ.get("JUDGE_MODE", DEFAULT_MODE)  # set "replay" or "live"
assert MODE in {"replay", "live"}
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from dotenv import load_dotenv
from judge_lab import (JudgeClient, batch, load_dataset, split_dataset, request_for,
                       metrics, select_threshold, run_cascade, needs_llm,
                       confidence_bands, always_false, save_json, digest,
                       POLICY, CRITERIA, JEV_MODEL, LLM_MODEL)

load_dotenv(LAB / ".env", override=False)
RUN = Path(os.environ.get("JUDGE_RUN_DIR", str(LAB / ".runs/judge")))
RUN.mkdir(parents=True, exist_ok=True)
CACHE = EVIDENCE / "cache" if MODE == "replay" else RUN / "cache"
if MODE == "replay" and not (EVIDENCE / "manifest.json").exists():
    raise RuntimeError("No recorded benchmark here. Run live with a funded key, then set JUDGE_EVIDENCE_DIR to that run directory.")
key = None
if MODE == "live":
    key = os.environ.get("OPENROUTER_API_KEY")
    if not key and IN_COLAB:
        try:
            from google.colab import userdata
            key = userdata.get("OPENROUTER_API_KEY")
        except Exception:
            pass
    if not key:
        from getpass import getpass
        key = getpass("OpenRouter key (hidden, never saved): ").strip()
client = JudgeClient(CACHE, live=(MODE == "live"), api_key=key)
print(f"Mode: {MODE}; Jev: {JEV_MODEL}; independent LLM: {LLM_MODEL}")
print("HTTP 401/402/403 stops the run. Correct credentials/credits, then resume from cache.")

## 2. Freeze the task, inputs and references

Both judges receive the same original intent definitions, vague-failure policy,
`USER_INPUT`, and the router's `final_intent`. The LLM does not see Jev's label or confidence.
Neither judge sees `TARGET_INTENT`, human comments, failure categories, or the human
`Vague_Utterance_Failure` label. Those fields would leak the answer.

The CSV contains 300 judge labels: 20 `TRUE` and 280 `FALSE`. A constant `FALSE` judge
therefore has 93.3% accuracy and zero failure recall. We report positive-class precision,
recall and F1, confusion counts, and unresolved outputs alongside accuracy.

The source files have historical inconsistencies. We preserve the explicit judge labels
and report **agreement with those labels**, not universal correctness. Cross-file intent
differences and a query mismatch are shown below. The legacy `intent_match` field contains
prose on many rows; we do not use it to derive the human judge label.

In [ ]:
rows = load_dataset(LAB / "data")
calibration, heldout = split_dataset(rows)
assert set(r["id"] for r in calibration).isdisjoint(r["id"] for r in heldout)
display(pd.DataFrame([{"split": name, "rows": len(part),
                       "TRUE labels": sum(r["human_label"] == "TRUE" for r in part)}
                      for name, part in [("Full dataset", rows), ("Calibration", calibration),
                                         ("Held out", heldout)]]))
audit = pd.DataFrame([{"ID": r["id"], "issue": issue}
                      for r in rows for issue in r["reference_issues"]])
display(audit.groupby("issue").size().rename("rows").to_frame())
audit.to_csv(RUN / "reference-audit.csv", index=False)
display(pd.DataFrame(rows[:3])[["id", "user_query", "predicted_intent", "human_label"]])
print("Dataset SHA-256:", digest(rows))

We reserve roughly one third of the examples for calibration and the rest for a final
held-out comparison. The split is deterministic, stratified by the human Boolean label,
and groups identical normalized queries together. Human labels balance the split; they
never enter an API request. We freeze the prompt before these calls and select the gate
using calibration data only.

In [ ]:
print("Shared policy:\n", POLICY)
display(pd.DataFrame([{"decision": label, "definition": text} for label, text in CRITERIA.items()]))
sample_request = request_for(rows[0], "jev")
print("State sent to Jev:", sample_request["state"])
assert set(sample_request["state"]) == {"user_query", "predicted_intent"}

## 3. Jev alone: a categorical judge

This task uses **`type: "choice"`**, with named `TRUE` / `FALSE` options. Jev returns
`choice`, `probabilities`, and `confidence`. We use its returned label directly. There is
no ordered quality scale, weighted-average `score`, or 1–5 conversion in this lab. A Noul
could represent the same yes/no question; Choice is used here to expose the confidence
field explicitly.

**Probability and confidence differ.** For a two-option Choice, TypeSafe defines
`confidence = 2 × max(probabilities) − 1`. Thus probabilities 0.85 / 0.15 yield confidence
0.70. A confidence of 0.70 does not mean 70% accuracy. We use the returned confidence
and check its relationship with errors on our own data.

[Choice documentation](https://docs.typesafe.ai/primitives/choice) ·
[Confidence definition](https://docs.typesafe.ai/confidence)

In [ ]:
jev_cal = batch(calibration, lambda r: client.call(r, "jev"))
example = calibration[0]
display(pd.DataFrame([jev_cal[example["id"]]]))
display(pd.DataFrame([metrics(calibration, jev_cal, "Jev alone — calibration")]))

## 4. LLM alone: the independent baseline

GPT-4.1-mini receives the same task and trace, with a JSON schema requiring `label` and a
brief `reason`. Schema validation catches malformed responses; it does not make an answer
correct. Temperature zero reduces sampling variation and does not guarantee repeatability.
The explanation is for inspection, not a promise that asking for reasoning improves accuracy.

This baseline is independent of Jev. The cascade will call **this same judge** on selected
traces, which lets us compare policies without changing the LLM's information or prompt.

In [ ]:
llm_cal = batch(calibration, lambda r: client.call(r, "llm"))
display(pd.DataFrame([llm_cal[example["id"]]]))
display(pd.DataFrame([metrics(calibration, values, name) for name, values in
                      [("Always FALSE", always_false(calibration)),
                       ("Jev alone", jev_cal), ("LLM alone", llm_cal)]]))

## 5. Choose the gate on calibration data

Proposed policy: keep a valid Jev judgment when **confidence ≥ threshold**; otherwise ask
the LLM. Invalid Jev output also escalates. If the LLM fails, the case remains unresolved
for human review; we do not silently replace it with a low-confidence Jev answer.

Before looking at held-out results, we compare thresholds 0, 0.25, 0.50, 0.70, 0.85, 0.95
and 1.00. The predeclared selection rule maximizes **TRUE-class F1**, then accuracy, then
minimizes LLM routes. The final tie goes to the lower threshold. The calibration set has
few positives, so this threshold is a workshop estimate, not a production recommendation.
Even threshold 1 retains valid Jev answers with confidence exactly 1.

In [ ]:
threshold, sweep = select_threshold(calibration, jev_cal, llm_cal)
display(pd.DataFrame(sweep)[["threshold", "accuracy", "precision_TRUE", "recall_TRUE",
                             "f1_TRUE", "llm_routes", "cost_usd", "unreported_cost_attempts"]])
print("Locked threshold:", threshold)
save_json(RUN / "calibration.json", {"threshold": threshold, "sweep": sweep,
                                    "ids": [r["id"] for r in calibration]})

## 6. Run the held-out cascade before the full LLM comparison

The next cell obtains held-out Jev judgments, then invokes the LLM **only** for the
escalated rows. The later LLM-alone benchmark fills in the remaining LLM results, reusing
the escalations already obtained. This separates the proposed deployment path from the
extra calls needed to benchmark it.

In [ ]:
jev_test = batch(heldout, lambda r: client.call(r, "jev"))
before_requests = client.new_requests
before_hits = client.cache_hits
cascade_test = run_cascade(heldout, client, threshold, jev=jev_test)
escalated = [r for r in heldout if needs_llm(jev_test[r["id"]], threshold)]
assert sum(r["route"] == "LLM" for r in cascade_test.values()) == len(escalated)
print(f"Escalated {len(escalated)}/{len(heldout)} held-out traces.")
print("Fresh HTTP attempts in gate:", client.new_requests - before_requests,
      "| replay/cache reads:", client.cache_hits - before_hits)
display(pd.DataFrame([metrics(heldout, cascade_test, "Jev then LLM")]))

In [ ]:
llm_test = batch(heldout, lambda r: client.call(r, "llm"))
comparison = pd.DataFrame([metrics(heldout, values, name) for name, values in
                          [("Always FALSE", always_false(heldout)), ("Jev alone", jev_test),
                           ("LLM alone", llm_test), ("Jev then LLM", cascade_test)]])
comparison["cost_per_1000_USD"] = comparison["cost_usd"] / len(heldout) * 1000
comparison["LLM_fraction"] = comparison["llm_routes"] / len(heldout)
display(comparison)
comparison.to_csv(RUN / "heldout-comparison.csv", index=False)

## 7. Compare accuracy and actual API costs

Costs come from each OpenRouter response's **`usage.cost` in USD**, including any recorded
retry responses. Missing cost records stay explicitly unknown; they are not free calls.
Replay makes no new API charges. The tables report the original recorded inference cost
for this dataset, not a price guarantee or the cost of executing cached cells.

For a deployed cascade:

`total cost = Jev cost on every trace + LLM cost on escalated traces`

If average costs were constant, this becomes `N × (c_Jev + escalation_rate × c_LLM)`.
The cascade is cheaper than LLM alone when `escalation_rate < 1 − c_Jev/c_LLM`.
The measured calculation below uses per-trace costs because longer prompts and responses
cost different amounts. The full research experiment also pays for calibration and the
independent LLM benchmark; that is separate from cascade deployment cost.

In [ ]:
plotted = comparison[comparison.method != "Always FALSE"].set_index("method")
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors = ["#4063d8", "#d88b30", "#26856b"]
for ax, field, title in zip(axes, ["accuracy", "f1_TRUE", "cost_per_1000_USD"],
                           ["Held-out agreement", "TRUE-class F1", "Recorded USD per 1,000 traces"]):
    if field.startswith("cost") and plotted.unreported_cost_attempts.sum():
        ax.text(.5, .5, "Costs incomplete: see table", ha="center")
    else:
        ax.bar(plotted.index, plotted[field], color=colors)
        if field != "cost_per_1000_USD": ax.set_ylim(0, 1)
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=20)
    ax.spines[["top", "right"]].set_visible(False)
axes[0].axhline(comparison.iloc[0].accuracy, color="gray", linestyle="--", label="Always FALSE")
axes[0].legend()
fig.tight_layout()
plt.show()
jev_cost = metrics(heldout, jev_test, "Jev")["cost_usd"]
llm_cost = metrics(heldout, llm_test, "LLM")["cost_usd"]
cascade_cost = metrics(heldout, cascade_test, "Cascade")["cost_usd"]
if not plotted.unreported_cost_attempts.sum() and llm_cost:
    print(f"Cascade cost change versus LLM alone: {(cascade_cost / llm_cost - 1):+.1%}")
else:
    print("Incomplete or zero reported costs: no savings claim.")
all_judges = [*jev_cal.values(), *llm_cal.values(), *jev_test.values(), *llm_test.values()]
print("Full benchmark recorded cost (USD):", sum(r["cost_usd"] for r in all_judges))
print("Unreported-cost attempts:", sum(r["unreported_cost_attempts"] for r in all_judges))

## 8. Why the cascade can work — and when it does not

The cost benefit comes from avoiding LLM calls on accepted Jev judgments. The accuracy
benefit requires something more: low confidence must identify cases worth reviewing,
and the LLM must correct more mistakes than it introduces on that subset. Simply adding
a second judge is not enough. A confidently wrong Jev decision bypasses the LLM; shared
prompt or label problems can confuse both models.

We therefore inspect the kept/escalated groups and count **LLM fixes versus regressions**.
This tests the mechanism instead of assuming confidence predicts accuracy. An LLM can
disagree with the human label and still have a defensible interpretation, so review the
original traces before changing the prompt or labels. Do not tune the threshold on the
held-out results shown below; use a new held-out set after further calibration.

In [ ]:
bands = confidence_bands(heldout, jev_test, llm_test, threshold)
display(pd.DataFrame(bands))
disagreements = []
for r in heldout:
    j, l, c = jev_test[r["id"]], llm_test[r["id"]], cascade_test[r["id"]]
    if j["label"] != l["label"] or c["label"] != r["human_label"]:
        disagreements.append({"ID": r["id"], "query": r["user_query"],
            "candidate_intent": r["predicted_intent"], "human": r["human_label"],
            "Jev": j["label"], "confidence": j.get("confidence"), "LLM": l["label"],
            "final": c["label"], "route": c["route"], "LLM_reason": l.get("reason"),
            "reference_issues": "; ".join(r["reference_issues"])})
display(pd.DataFrame(disagreements).head(12))
pd.DataFrame(disagreements).to_csv(RUN / "disagreements.csv", index=False)
# Show whether cross-file reference inconsistencies materially change the picture.
consistent = [r for r in heldout if not any("differs_between_csvs" in issue
                                           for issue in r["reference_issues"])]
display(pd.DataFrame([metrics(consistent, p, name + " — consistent references")
                      for name, p in [("Jev", jev_test), ("LLM", llm_test), ("Cascade", cascade_test)]]))

## 9. Historical LLM results and reproducibility checks

The original `judge_alignment_1.csv` and `judge_alignment_2.csv` remain available. Their
labels provide historical agreement baselines, but the files do not contain verified
per-request model metadata or token/cost records. We show their agreement without inventing
costs or treating them as controlled comparisons with this run. The fresh benchmark above
is the one that uses identical inputs, one frozen policy, and recorded costs.

In [ ]:
historical = []
for filename in ["judge_alignment_1.csv", "judge_alignment_2.csv"]:
    old = pd.read_csv(LAB / "data" / filename, dtype=str, keep_default_na=False)
    lookup = old.set_index("TEST_ID").to_dict("index")
    matching = [r for r in heldout if r["id"] in lookup
                and lookup[r["id"]]["USER_INPUT"] == r["user_query"]
                and lookup[r["id"]]["final_intent"] == r["predicted_intent"]
                and lookup[r["id"]]["Vague_Utterance_Failure"].upper() == r["human_label"]]
    predictions = {r["id"]: {"label": lookup[r["id"]]["LLM_Vague_Utterance_Failure"].upper()}
                   for r in matching}
    m = metrics(matching, predictions, filename)
    m["cost_usd"] = None  # Historical files do not record usage/cost.
    historical.append(m)
display(pd.DataFrame(historical)[["method", "n", "accuracy", "precision_TRUE", "recall_TRUE", "f1_TRUE", "cost_usd"]])

In [ ]:
assert len(jev_cal) + len(jev_test) == len(rows)
assert len(llm_cal) + len(llm_test) == len(rows)
assert len(cascade_test) == len(heldout)
assert set(r["id"] for r in calibration).isdisjoint(r["id"] for r in heldout)
if MODE == "replay":
    manifest = json.loads((EVIDENCE / "manifest.json").read_text())
    assert digest(rows) == manifest["dataset_hash"]
    assert digest([POLICY, CRITERIA]) == manifest["policy_hash"]
    assert [r["id"] for r in calibration] == manifest["calibration_ids"]
    assert [r["id"] for r in heldout] == manifest["heldout_ids"]
    assert threshold == manifest["threshold"]
    actual = [metrics(heldout, values, name) for name, values in
              [("Jev alone", jev_test), ("LLM alone", llm_test), ("Jev then LLM", cascade_test)]]
    assert actual == manifest["heldout_metrics"]
    assert client.new_requests == 0
if MODE == "live":
    save_json(RUN / "manifest.json", {
        "dataset_hash": digest(rows), "policy_hash": digest([POLICY, CRITERIA]),
        "calibration_ids": [r["id"] for r in calibration],
        "heldout_ids": [r["id"] for r in heldout], "threshold": threshold,
        "models": {"jev": JEV_MODEL, "llm": LLM_MODEL},
        "heldout_metrics": [metrics(heldout, values, name) for name, values in
                            [("Jev alone", jev_test), ("LLM alone", llm_test), ("Jev then LLM", cascade_test)]]})
    print("Replay this run by setting JUDGE_MODE=replay and JUDGE_EVIDENCE_DIR to", RUN)
save_json(RUN / "summary.json", {"mode": MODE, "threshold": threshold,
    "dataset_hash": digest(rows), "comparison": comparison.to_dict("records"),
    "bands": bands, "new_http_attempts": client.new_requests})
print("Notebook completed. Fresh HTTP attempts:", client.new_requests)
print("Outputs:", RUN)
print("Unresolved rows:", comparison.set_index("method")["unresolved"].to_dict())

## Take this into your own eval loop

Keep an explicit human rubric, a calibration set, and an untouched test set. Inspect rare
failure recall and high-confidence errors, not just aggregate accuracy. Choose the gate
for the product's cost of false alarms and missed failures. Recheck after model, prompt,
data, or pricing changes. Low-confidence fallback can improve cost and quality together,
but this notebook's measurements determine whether that happened on these traces.

To make a fresh run, select `live`, use a funded OpenRouter key, and choose a new
`JUDGE_RUN_DIR`. To replay that run, set `JUDGE_MODE=replay` and
`JUDGE_EVIDENCE_DIR` to the same directory. Changing the model or prompt changes the request hash, so stale cache
entries cannot silently supply results for a different experiment. Unit tests:
`python -m pytest tests -q` from this directory.